In [4]:
import pandas as pd
from squid.resources.trino import get_trino
from squid.jupyter.extensions import viewdf, viewdf_pandas

In [6]:
conn = get_trino()
cursor = conn.cursor()

In [7]:
def tsql(query: str):
    cursor.execute(query)
    columns = [column[0] for column in cursor.description]
    iceberg_system_tables = pd.DataFrame(cursor.fetchall(), columns=columns)
    display(iceberg_system_tables)
    

In [19]:
query = """
SELECT table_schema, table_name, table_type
FROM iceberg.information_schema.tables
ORDER BY table_schema, table_name
"""

tsql(query)

,table_schema,table_name,table_type
0,information_schema,applicable_roles,BASE TABLE
1,information_schema,columns,BASE TABLE
2,information_schema,enabled_roles,BASE TABLE
3,information_schema,roles,BASE TABLE
4,information_schema,schemata,BASE TABLE
5,information_schema,table_privileges,BASE TABLE
6,information_schema,tables,BASE TABLE
7,information_schema,views,BASE TABLE
8,nyc_gov,country_codes,BASE TABLE
9,nyc_gov,popular_names_by_country_2026,BASE TABLE


In [9]:
query = """
SELECT *
FROM iceberg.system.iceberg_tables
"""

tsql(query)

,table_schema,table_name


In [10]:
cursor.execute("""
CREATE SCHEMA IF NOT EXISTS iceberg.test
""")

In [13]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS iceberg.test.my_table (
    id BIGINT,
    name VARCHAR,
    amount DOUBLE,
    created_at TIMESTAMP
)
""")

In [14]:
query = """
SELECT *
FROM iceberg.system.iceberg_tables
"""

tsql(query)

,table_schema,table_name
0,test,my_table


In [15]:
cursor.execute("""
INSERT INTO iceberg.test.my_table
VALUES
    (1, 'Alice', 100.5, CURRENT_TIMESTAMP),
    (2, 'Bob',   200.0, CURRENT_TIMESTAMP)
""")

In [16]:
query = """
SELECT *
FROM iceberg.test.my_table
"""
tsql(query)

,id,name,amount,created_at
0,1,Alice,100.5,2026-09-04 19:01:18.079
1,2,Bob,200.0,2026-09-04 19:01:18.079
